# CDC Pipeline: Customer Info (Bronze → Silver → Gold)

**End-to-End Incremental Processing for `cust_info.csv`**

This notebook implements the complete CDC migration for customer information:

1. **Bronze Layer**: Auto Loader (incremental file ingestion)
2. **Silver Layer**: MERGE with transformations (upserts)
3. **Gold Layer**: Dimension table updates

**Benefits vs Previous Approach:**
* ✅ Only processes NEW files (Auto Loader)
* ✅ Only transforms NEW records (watermarking)
* ✅ Idempotent (safe to rerun)
* ✅ Preserves history (Delta versioning)
* ✅ Faster execution (incremental vs full scan)

---

**Tables:**
* Source: `/Volumes/workspace/bronze/raw_sources/source_crm/cust_info.csv`
* Bronze: `bronze.crm_cust_info_cdc`
* Silver: `silver.crm_customers_cdc`
* Gold: `gold.dim_customers` (existing table)

## Configuration

Set up paths and checkpoint locations.

In [0]:
# Configuration for CDC pipeline
from pyspark.sql.functions import current_timestamp, col, lit, trim, when, upper
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from delta.tables import DeltaTable

# Paths
source_path = "/Volumes/workspace/bronze/raw_sources/source_crm/cust_info.csv"
checkpoint_path = "/Volumes/workspace/bronze/checkpoints/cdc_crm_cust_info"

# Table names
bronze_table = "bronze.crm_cust_info_cdc"
silver_table = "silver.crm_customers_cdc"
gold_table = "gold.dim_customers"

print("✓ Configuration loaded")
print(f"  Source: {source_path}")
print(f"  Checkpoint: {checkpoint_path}")
print(f"  Bronze: {bronze_table}")
print(f"  Silver: {silver_table}")
print(f"  Gold: {gold_table}")

## Phase 1: Bronze Layer - Auto Loader (Incremental File Ingestion)

**What this does:**
* Discovers only NEW CSV files added since last run
* Tracks processed files in checkpoint directory
* Appends new records to bronze table
* Adds `ingestion_timestamp` for watermarking

**First run**: Processes all existing files  
**Subsequent runs**: Only processes new files

In [0]:
# Auto Loader: Read NEW files only
df_bronze_stream = (spark.readStream
  .format("cloudFiles")  # Auto Loader
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
  .option("header", "true")
  .option("inferSchema", "true")
  .load(source_path)
  .withColumn("ingestion_timestamp", current_timestamp())
)

print("✓ Auto Loader configured")
print("  Processing mode: Incremental (new files only)")
print("  Schema evolution: Enabled")

In [0]:
# Write stream to Bronze table
# Only NEW files will be processed
print("Starting Auto Loader ingestion...")

(df_bronze_stream.writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", checkpoint_path)
  .option("mergeSchema", "true")
  .trigger(availableNow=True)  # Batch mode: process and stop
  .table(bronze_table)
  .awaitTermination()
)

print("\n✓ Bronze ingestion complete")
print("✓ Checkpoint saved - next run will skip these files")

In [0]:
%sql
-- Verify bronze table
SELECT 
  COUNT(*) as total_records,
  COUNT(DISTINCT cst_id) as unique_customers,
  MIN(ingestion_timestamp) as first_ingestion,
  MAX(ingestion_timestamp) as last_ingestion
FROM bronze.crm_cust_info_cdc

## Phase 2: Silver Layer - Transformations + MERGE (Upserts)

**What this does:**
* Watermarking: Only reads NEW records from bronze (by `ingestion_timestamp`)
* Transformations: Same logic as your existing silver notebook
  - Trimming string columns
  - Normalizing marital status (S→Single, M→Married)
  - Normalizing gender (F→Female, M→Male)
  - Filtering null customer IDs
  - Renaming columns
* MERGE: Updates existing + inserts new (idempotent)

**First run**: Processes all bronze data  
**Subsequent runs**: Only processes records added since last run

In [0]:
# Watermarking: Only process NEW data
try:
    last_processed = spark.sql(f"""
        SELECT MAX(ingestion_timestamp) as watermark 
        FROM {silver_table}
    """).collect()[0][0]
    
    if last_processed is None:
        print("✓ First run - processing all bronze data")
        last_processed = "1900-01-01"
    else:
        print(f"✓ Watermark found: {last_processed}")
        print(f"✓ Processing only records after this timestamp")
except:
    print("✓ Silver table doesn't exist - will create on first MERGE")
    last_processed = "1900-01-01"

# Read only NEW records from bronze
new_records = (spark.table(bronze_table)
  .where(col("ingestion_timestamp") > lit(last_processed))
)

record_count = new_records.count()
print(f"\n✓ Found {record_count:,} new records to process")

In [0]:
# Apply silver transformations (same as existing logic)
if record_count > 0:
    # 1. Trimming
    for field in new_records.schema.fields:
        if isinstance(field.dataType, StringType):
            new_records = new_records.withColumn(field.name, trim(col(field.name)))
    
    # 2. Normalization
    new_records = (
        new_records
        .withColumn(
            "cst_marital_status",
            when(upper(col("cst_marital_status")) == "S", "Single")
            .when(upper(col("cst_marital_status")) == "M", "Married")
            .otherwise("n/a")
        )
        .withColumn(
            "cst_gndr",
            when(upper(col("cst_gndr")) == "F", "Female")
            .when(upper(col("cst_gndr")) == "M", "Male")
            .otherwise("n/a")
        )
    )
    
    # 3. Remove nulls
    new_records = new_records.filter(col("cst_id").isNotNull())
    
    # 4. Rename columns
    rename_map = {
        "cst_id": "customer_id",
        "cst_key": "customer_number",
        "cst_firstname": "first_name",
        "cst_lastname": "last_name",
        "cst_marital_status": "marital_status",
        "cst_gndr": "gender",
        "cst_create_date": "created_date"
    }
    
    for old_name, new_name in rename_map.items():
        new_records = new_records.withColumnRenamed(old_name, new_name)
    
    final_count = new_records.count()
    print(f"✓ Transformations applied")
    print(f"  Input: {record_count:,} records")
    print(f"  Output: {final_count:,} records (after null filtering)")
else:
    print("✓ No new records to process - skipping transformations")

In [0]:
# MERGE: Update existing + Insert new (idempotent)
if record_count > 0:
    # Create table if doesn't exist
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {silver_table} (
            customer_id INT,
            customer_number STRING,
            first_name STRING,
            last_name STRING,
            marital_status STRING,
            gender STRING,
            created_date DATE,
            ingestion_timestamp TIMESTAMP
        ) USING DELTA
    """)
    
    # Perform MERGE
    silver_table_delta = DeltaTable.forName(spark, silver_table)
    
    (silver_table_delta.alias("target")
      .merge(
          new_records.alias("source"),
          "target.customer_id = source.customer_id"
      )
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute()
    )
    
    print("\n✓ MERGE completed successfully")
    print("  Existing customers: UPDATED")
    print("  New customers: INSERTED")
    print("\n💡 This operation is IDEMPOTENT - safe to rerun")
else:
    print("✓ No MERGE needed - no new records")

In [0]:
%sql
-- Verify silver table after MERGE
SELECT 
  COUNT(*) as total_customers,
  COUNT(DISTINCT customer_id) as unique_customers,
  MAX(ingestion_timestamp) as last_update,
  COUNT(CASE WHEN marital_status = 'n/a' THEN 1 END) as unknown_marital,
  COUNT(CASE WHEN gender = 'n/a' THEN 1 END) as unknown_gender
FROM silver.crm_customers_cdc

## Phase 3: Gold Layer - Dimension Table Update

**What this does:**
* Reads transformed customer data from silver
* MERGES into existing `gold.dim_customers` table
* Uses silver's `customer_id` as the business key
* Updates existing dimensions, inserts new ones

**Note**: This updates your EXISTING gold dimension table

In [0]:
# Read from silver (only what we just processed/updated)
silver_df = spark.table(silver_table)

# Transform for gold schema (add customer_key as surrogate key)
gold_source = (
    silver_df
    .withColumn("customer_key", col("customer_id"))  # Use customer_id as key
    .select(
        "customer_key",
        "customer_id",
        "customer_number",
        "first_name",
        "last_name",
        "marital_status",
        "gender",
        "created_date"
    )
)

print(f"✓ Prepared {gold_source.count():,} records for gold dimension")

In [0]:
# Check if gold table exists, create if not
try:
    gold_table_delta = DeltaTable.forName(spark, gold_table)
    print("✓ Gold dimension table exists")
except:
    print("✓ Creating gold dimension table...")
    spark.sql(f"""
        CREATE TABLE {gold_table} (
            customer_key INT,
            customer_id INT,
            customer_number STRING,
            first_name STRING,
            last_name STRING,
            marital_status STRING,
            gender STRING,
            created_date DATE
        ) USING DELTA
    """)
    gold_table_delta = DeltaTable.forName(spark, gold_table)

# MERGE into gold
(gold_table_delta.alias("target")
  .merge(
      gold_source.alias("source"),
      "target.customer_key = source.customer_key"
  )
  .whenMatchedUpdateAll()
  .whenNotMatchedInsertAll()
  .execute()
)

print("\n✓ Gold dimension updated successfully")
print("✓ Pipeline complete: Bronze → Silver → Gold")

In [0]:
%sql
-- Verify gold dimension
SELECT 
  COUNT(*) as total_customers,
  COUNT(DISTINCT customer_key) as unique_keys,
  COUNT(CASE WHEN marital_status = 'Married' THEN 1 END) as married,
  COUNT(CASE WHEN marital_status = 'Single' THEN 1 END) as single,
  COUNT(CASE WHEN gender = 'Male' THEN 1 END) as male,
  COUNT(CASE WHEN gender = 'Female' THEN 1 END) as female
FROM gold.dim_customers

## Testing: Idempotency Check

Run this cell to verify the pipeline is idempotent (produces same results when run multiple times).

In [0]:
# Test idempotency: Run pipeline twice, compare counts
print("=" * 60)
print("IDEMPOTENCY TEST")
print("=" * 60)

# Get current counts
silver_count_1 = spark.table(silver_table).count()
gold_count_1 = spark.table(gold_table).count()

print(f"\nBefore re-run:")
print(f"  Silver: {silver_count_1:,} records")
print(f"  Gold: {gold_count_1:,} records")

print("\n⏳ Re-running pipeline (should find 0 new records)...\n")

# Re-run pipeline
# 1. Auto Loader (should find no new files)
(spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
  .option("header", "true")
  .option("inferSchema", "true")
  .load(source_path)
  .withColumn("ingestion_timestamp", current_timestamp())
  .writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", checkpoint_path)
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .table(bronze_table)
  .awaitTermination()
)

print("✓ Auto Loader re-run complete\n")

# 2. Watermarking (should find no new records)
last_processed = spark.sql(f"SELECT MAX(ingestion_timestamp) FROM {silver_table}").collect()[0][0]
new_records_test = spark.table(bronze_table).where(col("ingestion_timestamp") > lit(last_processed))
print(f"✓ Watermark check: {new_records_test.count()} new records found\n")

# Get counts after re-run
silver_count_2 = spark.table(silver_table).count()
gold_count_2 = spark.table(gold_table).count()

print(f"After re-run:")
print(f"  Silver: {silver_count_2:,} records")
print(f"  Gold: {gold_count_2:,} records")

print("\n" + "=" * 60)
if silver_count_1 == silver_count_2 and gold_count_1 == gold_count_2:
    print("✅ IDEMPOTENCY TEST PASSED")
    print("   Pipeline is safe to rerun - no duplicates created")
else:
    print("❌ IDEMPOTENCY TEST FAILED")
    print(f"   Silver diff: {silver_count_2 - silver_count_1}")
    print(f"   Gold diff: {gold_count_2 - gold_count_1}")
print("=" * 60)

## Pipeline Summary

### What We Built
✅ **Bronze**: Auto Loader ingests only NEW CSV files  
✅ **Silver**: Watermarking + MERGE processes only NEW records  
✅ **Gold**: Dimension table updated incrementally  
✅ **Idempotent**: Safe to rerun without duplicates  

### Performance Gains
**Before (Full Overwrite):**
* Bronze: Reads entire directory every time
* Silver: Processes all bronze records every time
* Gold: Replaces entire dimension every time

**After (CDC):**
* Bronze: Reads only new files (checkpoint tracking)
* Silver: Processes only new records (watermarking)
* Gold: Updates only changed dimensions (MERGE)

**Expected Improvement**: 10x-100x faster on subsequent runs

---

### Next Steps

1. **Test with new files**: Add a new CSV file to source directory, rerun notebook
2. **Monitor checkpoint**: Check `/Volumes/workspace/bronze/checkpoints/cdc_crm_cust_info/`
3. **Migrate other tables**: Apply same pattern to `prd_info`, `sales_details`, etc.
4. **Schedule job**: Set up daily/hourly runs in Databricks Jobs
5. **Compare costs**: Track compute costs vs old full-overwrite approach

### Troubleshooting

**If no records processed**:
* Check source path has CSV files
* Verify checkpoint location is writable
* Check watermark timestamp (shouldn't be in future)

**If duplicates appear**:
* Verify business key (customer_id) is unique
* Check MERGE condition matches on correct column
* Run idempotency test cell

**If schema evolution breaks**:
* Ensure `.option("mergeSchema", "true")` is set
* Check schema location in checkpoint directory
* Review Auto Loader schema inference settings